# Paper-ready steering examples — ICA basis (Llama-3.1-8B-Instruct, layer 19)

This notebook generates side-by-side text completions under steering of selected
ICA basis components (`b4`, `b7`, `b8`) discovered in Phase C-2. The output is
intended to be lifted directly into the paper as a qualitative table.

* Basis: `data/emotion_code/basis_sweep/ica_k016_seed0.pt` (ICA, k=16, layer 19)
* Steering: residual-stream additive hook, $\alpha \cdot b_j$ injected at layer 19
* Effective scale: $\alpha_{\text{eff}} = \alpha_{\text{unit}} \cdot \frac{1}{\text{median}_j \|b_j\|} \cdot \|b_{j^\star}\|$ (matches `eval_basis_*` scripts)

Outputs are saved to `artifacts/paper_examples/` as Markdown and LaTeX (`tabularx`).

## 1. Load model and basis

In [ ]:
from pathlib import Path
import sys, os

# Resolve the repo root robustly: try (1) walking up from cwd, (2) known
# absolute candidate. Idempotent so re-running cells does not escape the tree.
def _find_root_from(start: Path) -> Path | None:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists():
            return p
    return None

ROOT = (
    _find_root_from(Path.cwd())
    or _find_root_from(Path("/home/maplesugano/proj/EmotionEngine"))
)
if ROOT is None:
    raise RuntimeError("could not locate repo root (set REPO_ROOT env var)")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print(f"cwd: {os.getcwd()}")

import json
import numpy as np
import pandas as pd
import torch
import yaml

from src.activations._runtime import load_profile, load_model
from src.steering.generate import steered_generate
from experiments._gen_cache import load_neutral_prompts

BASIS_PATH = Path("data/emotion_code/basis_sweep/ica_k016_seed0.pt")
OUT_DIR = Path("artifacts/paper_examples")
OUT_DIR.mkdir(parents=True, exist_ok=True)

payload = torch.load(BASIS_PATH, weights_only=False, map_location="cpu")
which = payload.get("decomposer", "ica")
W = payload[which]["W"].numpy().astype(np.float32)   # [k, D]
LAYER = int(payload["layer"])
norms = np.linalg.norm(W, axis=1)
scale = 1.0 / float(np.median(norms))
print(f"basis k={W.shape[0]} D={W.shape[1]} layer={LAYER} scale={scale:.4f}")

RuntimeError: could not locate repo root

In [ ]:
profile, _ = load_profile(Path("configs/model.yaml"))
sc = yaml.safe_load(Path("configs/steering.yaml").read_text())
APPLY_TO = sc["caa"].get("apply_to", "generation")
model, device, _ = load_model(profile)

## 2. Choose prompts and components

Prompts are drawn from the same neutral DailyDialog set used by the
self-consistency evaluation, so the qualitative table is comparable to the
quantitative numbers in the paper. We select 5 prompts with diverse register
(question, statement, request, narration, hospitality).

In [ ]:
ALL_PROMPTS = load_neutral_prompts(n=12, seed=0)
PROMPTS = ALL_PROMPTS[:5]
for i, p in enumerate(PROMPTS):
    print(f"[{i}] {p}")

In [ ]:
# Components: the three Phase C-2 candidates.
COMPONENTS = {
    8: "b8 — addressivity vs analytical detachment (emergent at L19)",
    4: "b4 — irony / detachment",
    7: "b7 — fragmentation (likely artefact, kept for contrast)",
}
# Finer-grained α grid: 7 points symmetric around 0.
# 0 = baseline; ±1 stays close to baseline; ±3 near saturation onset;
# ±6 fully saturated (revealed the axis qualitatively in earlier runs).
ALPHAS = [-6.0, -3.0, -1.0, 0.0, +1.0, +3.0, +6.0]
MAX_NEW_TOKENS = 60

## 3. Generate the steered completions

In [ ]:
from tqdm.auto import tqdm
rows = []
for j, label in COMPONENTS.items():
    b_j = torch.from_numpy(W[j]).to(torch.float32)
    for pi, prompt in enumerate(tqdm(PROMPTS, desc=f"b{j}")):
        for a in ALPHAS:
            if a == 0.0:
                vec, alpha = torch.zeros_like(b_j), 0.0
            else:
                vec, alpha = b_j, float(a) * scale * float(norms[j])
            out = steered_generate(
                model, prompt, vector=vec, alpha=alpha,
                layers=[LAYER], apply_to=APPLY_TO,
                max_new_tokens=MAX_NEW_TOKENS,
            )
            tail = out[len(prompt):].strip() if out.startswith(prompt) else out.strip()
            rows.append({
                "component": j, "label": label,
                "prompt_id": pi, "prompt": prompt,
                "alpha": a, "generation": tail,
            })
df = pd.DataFrame(rows)
df.to_parquet(OUT_DIR / "b8_b4_b7_examples.parquet", index=False)
df.head(3)

## 4. Render side-by-side Markdown table

One section per component. Each row = one prompt; columns = $\alpha \in \{-6, 0, +6\}$.

In [ ]:
from IPython.display import Markdown, display

def truncate(s: str, n: int = 220) -> str:
    s = s.replace("\n", " ").strip()
    return s if len(s) <= n else s[:n - 1].rstrip() + "…"

def _alpha_label(a: float) -> str:
    if a == 0.0:
        return "$\\alpha = 0$ (baseline)"
    return f"$\\alpha = {a:+g}$"

header = "| Prompt | " + " | ".join(_alpha_label(a) for a in ALPHAS) + " |"
sep    = "|---|" + "|".join(["---"] * len(ALPHAS)) + "|"
lines = ["# Steering examples (ICA basis, Llama-3.1-8B-Instruct, layer 19)\n"]
for j, label in COMPONENTS.items():
    lines += [f"\n## Component `b{j}` — {label}\n", header, sep]
    sub = df[df.component == j]
    for pi in sorted(sub.prompt_id.unique()):
        row = sub[sub.prompt_id == pi]
        prm = truncate(row.iloc[0].prompt, 80)
        cells = [
            truncate(row[row.alpha == a].iloc[0].generation, 180).replace("|", "\\|")
            for a in ALPHAS
        ]
        lines.append("| _" + prm + "_ | " + " | ".join(cells) + " |")
md = "\n".join(lines)
(OUT_DIR / "b8_b4_b7_examples.md").write_text(md)
display(Markdown(md))

## 5. LaTeX export (`tabularx`) for the paper

One `table*` per component, ready to `\input{}` into the paper. Uses
`tabularx` so the prompt and three generation columns share width.

In [ ]:
def latex_escape(s: str) -> str:
    repl = {
        "\\": r"\textbackslash{}", "&": r"\&", "%": r"\%", "$": r"\$",
        "#": r"\#", "_": r"\_", "{": r"\{", "}": r"\}",
        "~": r"\textasciitilde{}", "^": r"\textasciicircum{}",
    }
    out = []
    for ch in s:
        out.append(repl.get(ch, ch))
    return "".join(out)

def _tex_alpha(a: float) -> str:
    if a == 0.0:
        return r"$\alpha=0$"
    return rf"$\alpha={a:+g}$"

N_ALPHA = len(ALPHAS)
colspec = r"p{0.14\textwidth} " + " ".join(["X"] * N_ALPHA)
header_tex = "Prompt & " + " & ".join(_tex_alpha(a) for a in ALPHAS) + r" \\"

for j, label in COMPONENTS.items():
    sub = df[df.component == j]
    body = []
    body.append(r"\begin{table*}[t]")
    body.append(r"\centering\scriptsize")
    body.append(rf"\caption{{Steered generations for ICA component \texttt{{b{j}}} ({latex_escape(label.split('—')[1].strip())}). Layer 19, $\alpha$ in units of $1/\mathrm{{median}}\|b_j\|$.}}")
    body.append(rf"\label{{tab:steer_b{j}}}")
    body.append(rf"\begin{{tabularx}}{{\textwidth}}{{{colspec}}}")
    body.append(r"\toprule")
    body.append(header_tex)
    body.append(r"\midrule")
    for pi in sorted(sub.prompt_id.unique()):
        row = sub[sub.prompt_id == pi]
        prm = latex_escape(truncate(row.iloc[0].prompt, 80))
        cells = [latex_escape(truncate(row[row.alpha == a].iloc[0].generation, 200))
                 for a in ALPHAS]
        body.append(rf"\textit{{{prm}}} & " + " & ".join(cells) + r" \\")
    body.append(r"\bottomrule")
    body.append(r"\end{tabularx}")
    body.append(r"\end{table*}")
    tex = "\n".join(body) + "\n"
    path = OUT_DIR / f"steering_b{j}.tex"
    path.write_text(tex)
    print(f"wrote {path}")

## 6. Numeric summary (Phase C-2 metrics for the same three components)

In [ ]:
summ = pd.read_csv("experiments/results/basis_selfconsistency_full/ica_k016_seed0__summary.csv")
mono = pd.read_csv("experiments/results/basis_selfconsistency_full/ica_k016_seed0__monotonicity.csv")
metr = pd.read_csv("data/emotion_code/basis_sweep/metrics.summary.csv")
metr = metr[(metr.decomposer == "ica") & (metr.k == 16) & (metr.seed == 0)]
lc = pd.read_csv("data/emotion_code/basis_sweep/layer_component_consistency.csv")
rows = []
for j in COMPONENTS:
    s = summ[summ.component == j]
    pos = s[s.alpha_unit == 2.0].iloc[0]
    neg = s[s.alpha_unit == -2.0].iloc[0]
    m = mono[mono.component == j].iloc[0]
    me = metr[metr.component == j].iloc[0]
    # mean abs_cosine across L19 ↔ {L13, L16, L22}
    vals = []
    for L_other in [13, 16, 22]:
        if L_other < 19:
            sub = lc[(lc.layer_a == L_other) & (lc.layer_b == 19) & (lc.component_b == j)]
        else:
            sub = lc[(lc.layer_a == 19) & (lc.layer_b == L_other) & (lc.component_a == j)]
        if not sub.empty:
            vals.append(float(sub.iloc[0].abs_cosine))
    rows.append({
        "component": f"b{j}",
        "self_+2": round(float(pos.self_delta_cosine), 3),
        "self_-2": round(float(neg.self_delta_cosine), 3),
        "rho_self": round(float(m.spearman_self_mean), 3),
        "label_dom": round(float(me.category_top1_dominance), 3),
        "label_MI": round(float(me.mi), 3),
        "layer_consistency": round(float(np.mean(vals)), 3) if vals else None,
    })
metric_df = pd.DataFrame(rows)
metric_df.to_csv(OUT_DIR / "b8_b4_b7_metrics.csv", index=False)
metric_df